# Semantic — the dashboard views

Five views, each a join and a `GROUP BY` over Gold. Nothing is recalculated.

| view | grain | what the dashboard does with it |
|---|---|---|
| `v_beat_rate` | horizon × cohort | **the headline** — beat rate, twice per horizon |
| `v_leaderboard` | ticker × horizon | best and worst, by return, income or risk — **SPY ranked in** |
| `v_beat_rate_by_cut` | cut × value × horizon | the three cuts: management group, sole/multi, sector |
| `v_growth_of_100` | ticker × month | the line chart: what £100 became |
| `v_universe` | ticker | every trust in scope, and why each is or is not in the results |

**The leaderboard never feeds the beat rate.** Rank to a top ten and *then* ask "what share
beat the index?" and the answer is always 100%, because you picked them for winning. The
beat rate reads the whole fact; the leaderboard is display only.

In [0]:
-- THE HEADLINE. Beat rate by horizon, twice: everything, and survivors only.
-- Survivorship is a WHERE clause on the dimension, not a second fact table.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate
COMMENT 'Beat rate and risk by horizon, including and excluding delisted trusts'
AS
WITH spy AS (
  SELECT horizon_years, volatility AS spy_volatility, total_return AS spy_return
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
  WHERE ticker = 'SPY'
),
trusts AS (
  SELECT f.horizon_years, f.ticker, f.total_return, f.volatility,
         f.risk_adjusted_return, f.beat_index, d.status
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
),
cohorts AS (
  SELECT 'all trusts'     AS cohort, horizon_years, ticker, total_return, volatility,
         risk_adjusted_return, beat_index FROM trusts
  UNION ALL
  SELECT 'survivors only', horizon_years, ticker, total_return, volatility,
         risk_adjusted_return, beat_index FROM trusts WHERE status = 'active'
)
SELECT c.horizon_years,
       c.cohort,
       COUNT(*)                                                      AS trusts,
       SUM(CASE WHEN c.beat_index THEN 1 ELSE 0 END)                 AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN c.beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                          AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(c.total_return, 0.5), 1)        AS median_return_pct,
       ROUND(100 * MAX(s.spy_return), 1)                             AS index_return_pct,
       ROUND(100 * PERCENTILE_APPROX(c.volatility, 0.5), 1)          AS median_volatility_pct,
       ROUND(100 * MAX(s.spy_volatility), 1)                         AS index_volatility_pct,
       ROUND(100.0 * SUM(CASE WHEN c.volatility < s.spy_volatility
                              THEN 1 ELSE 0 END) / COUNT(*), 1)      AS calmer_than_index_pct,
       -- The only unambiguous win: beat it AND took less risk doing so.
       ROUND(100.0 * SUM(CASE WHEN c.beat_index AND c.volatility < s.spy_volatility
                              THEN 1 ELSE 0 END) / COUNT(*), 1)      AS beat_and_calmer_pct
FROM cohorts c
JOIN spy s ON s.horizon_years = c.horizon_years
GROUP BY c.horizon_years, c.cohort;

In [0]:
-- THE LEADERBOARD. Every ticker at every horizon, with SPY ranked among them so its
-- position is always visible. Display only: the dashboard sorts and limits this view, and
-- the beat rate above never reads it. Ranking the fact and then filtering it would make
-- "what % beat the index" circular -- the answer would always be 100%.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_leaderboard
COMMENT 'Every ticker at every horizon, ranked by return, income and risk-adjusted return'
AS
SELECT f.horizon_years,
       f.ticker,
       CASE WHEN d.entity_type = 'Index' THEN CONCAT(d.trust_name, ' (the index)')
            ELSE d.trust_name END                        AS name,
       d.entity_type,
       d.management_group,
       d.manager,
       d.manager_structure,
       d.aic_sector,
       d.status,
       f.rank_by_return,
       f.rank_by_income,
       f.rank_by_risk_adjusted,
       ROUND(100 * f.total_return, 1)                    AS total_return_pct,
       ROUND(100 * f.annualised_return, 2)               AS annualised_return_pct,
       ROUND(100 * f.income_return, 1)                   AS income_pct,
       ROUND(100 * f.volatility, 1)                      AS volatility_pct,
       ROUND(f.risk_adjusted_return, 2)                  AS risk_adjusted_return,
       ROUND(100 * f.index_return_same_period, 1)        AS index_return_pct,
       f.beat_index,
       f.months_used,
       f.return_basis,
       d.source_url
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key;

In [0]:
-- THE THREE CUTS. One view, one extra column saying which cut a row belongs to, because
-- three near-identical views would be three things to maintain and explain.
-- Every one is a GROUP BY over a column dim_ticker already holds. No model changes.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
COMMENT 'Beat rate by management group, by sole/multi manager, and by AIC sector'
AS
WITH trusts AS (
  SELECT f.horizon_years, f.beat_index, f.total_return, f.volatility,
         f.risk_adjusted_return,
         d.management_group, d.manager_structure, d.aic_sector
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
),
cut AS (
  SELECT 'management group' AS cut_by, management_group  AS cut_value, * FROM trusts
  UNION ALL
  SELECT 'manager structure', manager_structure, * FROM trusts
  UNION ALL
  SELECT 'aic sector', aic_sector, * FROM trusts
)
SELECT cut_by,
       cut_value,
       horizon_years,
       COUNT(*)                                                   AS trusts,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)                AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                       AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(total_return, 0.5), 1)       AS median_return_pct,
       ROUND(100 * PERCENTILE_APPROX(volatility, 0.5), 1)         AS median_volatility_pct,
       ROUND(PERCENTILE_APPROX(risk_adjusted_return, 0.5), 2)     AS median_risk_adjusted
FROM cut
GROUP BY cut_by, cut_value, horizon_years;

In [0]:
-- THE CHART. What £100 would have become, month by month, for every ticker.
-- A running product in log space again: adding logarithms is multiplying, and SQL has no
-- running product. This is the one view the dashboard plots as a line.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_growth_of_100
COMMENT 'Value of 100 invested at the start of the study window, monthly, per ticker'
AS
SELECT f.ticker,
       CASE WHEN d.entity_type = 'Index' THEN CONCAT(d.trust_name, ' (the index)')
            ELSE d.trust_name END                      AS name,
       d.entity_type,
       d.management_group,
       d.status,
       f.month_key,
       dd.month_start,
       dd.month_label,
       dd.market_event,
       ROUND(100 * f.total_return, 2)                  AS monthly_return_pct,
       ROUND(100 * EXP(SUM(LN(1 + COALESCE(f.total_return, 0))) OVER (
                       PARTITION BY f.ticker ORDER BY f.month_key
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)), 2)
                                                       AS value_of_100
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key
JOIN `index-vs-trust-pipeline`.gold.dim_date   dd ON dd.month_key = f.month_key
WHERE f.return_basis = 'total';

In [0]:
-- THE UNIVERSE. Every trust that was ever in scope, including the ones with no prices.
-- This is the survivorship evidence and the honesty of the study in one table: a pipeline
-- that only loaded what came back would not know the erased trusts ever existed.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_universe
COMMENT 'All 118 trusts and 3 index trackers, with why each is or is not in the results'
AS
SELECT d.ticker,
       d.trust_name,
       d.entity_type,
       d.management_group,
       d.manager_structure,
       d.aic_sector,
       d.status,
       d.data_status,
       d.price_source,
       d.months_available,
       d.first_month,
       d.last_month,
       d.source_url,
       CASE WHEN d.data_status = 'no-data'  THEN 'Yahoo returns nothing for it'
            WHEN d.data_status = 'excluded' THEN 'price history could not be trusted'
            WHEN d.data_status = 'stub'     THEN 'under 36 months of history'
            ELSE 'in the results'
       END                                            AS why,
       CASE WHEN d.data_status = 'usable' THEN true ELSE false END AS counts_in_the_beat_rate
FROM `index-vs-trust-pipeline`.gold.dim_ticker d
WHERE d.is_current;

## Verification

In [0]:
SELECT horizon_years, cohort, trusts, beat_count, beat_rate_pct,
       median_volatility_pct, index_volatility_pct,
       calmer_than_index_pct, beat_and_calmer_pct
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate
ORDER BY horizon_years DESC, cohort;

Expect **10 rows** — five horizons, two cohorts each.

`all trusts` should read **5.3 / 10.6 / 15.6 / 30.0 / 41.6**, and `beat_and_calmer_pct` must
be **0.0 at both 15 and 10 years**. The two cohorts differ by under one percentage point,
because only one or two delisted trusts are recoverable — report that as a demonstrated
mechanism, never as a measured effect size.

In [0]:
-- Where the index sits on each leaderboard, and the top five it is competing with.
SELECT horizon_years, ticker, name, rank_by_return, rank_by_income,
       rank_by_risk_adjusted, total_return_pct, income_pct, risk_adjusted_return
FROM `index-vs-trust-pipeline`.semantic.v_leaderboard
WHERE horizon_years = 10 AND (ticker = 'SPY' OR rank_by_return <= 5)
ORDER BY rank_by_return;

Expect **6 rows**: the top five by return, plus SPY at **rank 11 on return, 58 on income and
3 on risk-adjusted** — the two above it on risk being IVV and VOO, which track the same
index. No trust beats it once risk is counted.

In [0]:
-- Do the big houses do better? The strongest of the three cuts.
SELECT cut_value AS management_group, trusts, beat_count, beat_rate_pct,
       median_return_pct, median_volatility_pct
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
WHERE cut_by = 'management group' AND horizon_years = 10 AND trusts >= 4
ORDER BY beat_rate_pct DESC, trusts DESC;

Only groups with **4 or more trusts** are shown, because a 100% beat rate from one trust is
noise dressed as a finding. Always put the count beside the rate.

In [0]:
-- Every view returns rows, and the universe still adds up to the full 121.
SELECT 'v_beat_rate'        AS view_name, COUNT(*) AS rows FROM `index-vs-trust-pipeline`.semantic.v_beat_rate
UNION ALL SELECT 'v_leaderboard',       COUNT(*) FROM `index-vs-trust-pipeline`.semantic.v_leaderboard
UNION ALL SELECT 'v_beat_rate_by_cut',  COUNT(*) FROM `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
UNION ALL SELECT 'v_growth_of_100',     COUNT(*) FROM `index-vs-trust-pipeline`.semantic.v_growth_of_100
UNION ALL SELECT 'v_universe',          COUNT(*) FROM `index-vs-trust-pipeline`.semantic.v_universe
ORDER BY view_name;

Expect **v_beat_rate 10**, **v_leaderboard 445**, **v_universe 121**, and the other two
non-zero. `v_universe` at 121 is the check that the honest universe survives all the way to
the dashboard — 118 trusts plus the three trackers, including the ones with no prices.